# 特定投資株式の時価総額比率分析

このnotebookは、有価証券報告書提出企業の時価総額と、その企業が保有する特定投資株式の合計時価総額の割合を分析します。

## 1. セットアップ

In [ ]:
# 必要なライブラリのインポート
import os
import sys
import pandas as pd
import numpy as np
import yfinance as yf
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import time

warnings.filterwarnings('ignore')

# 日本語フォント設定
plt.rcParams['font.family'] = 'DejaVu Sans'

# 既存のスクリプトをインポート
sys.path.append('/Users/satoki252595/work/0000_kabulab/0001_MarketableSecuritiesAnalysis')
from improved_extraction import ImprovedMarketableSecuritiesExtractor

print("セットアップ完了")

## 2. データ読み込み（既存のCSVまたは新規抽出）

In [ ]:
# データ読み込み設定
USE_EXISTING_CSV = True  # 既存のCSVを使用する場合はTrue
CSV_FILE = "improved_marketable_securities_analysis.csv"  # 既存のCSVファイル名
XBRL_FOLDER_PATH = "/Users/satoki252595/work/0000_kabulab/0001_MarketableSecuritiesAnalysis/xbrl"
LIMIT_FOLDERS = 50  # 新規抽出時のフォルダ数制限

if USE_EXISTING_CSV and os.path.exists(CSV_FILE):
    # 既存のCSVファイルを読み込み
    print(f"既存のCSVファイルを読み込みます: {CSV_FILE}")
    df = pd.read_csv(CSV_FILE, encoding='utf-8-sig')
    print(f"読み込み完了: {len(df)} レコード")
else:
    # 新規データ抽出
    print("新規データ抽出を開始します...")
    extractor = ImprovedMarketableSecuritiesExtractor(XBRL_FOLDER_PATH)
    extractor.process_all_folders(limit=LIMIT_FOLDERS)
    df = extractor.to_dataframe()
    
    if not df.empty:
        df.to_csv("market_cap_analysis_data.csv", index=False, encoding='utf-8-sig')
        print(f"データ抽出完了: {len(df)} レコード")
    else:
        print("データが抽出されませんでした")

# データ確認
if not df.empty:
    print(f"\n提出会社数: {df['filing_company_code'].nunique()}")
    print(f"保有銘柄数: {df['held_security_name'].nunique()}")
    display(df.head())

## 3. 提出会社の時価総額取得

In [ ]:
def get_market_cap(stock_code, retry_count=3):
    """株式コードから時価総額を取得"""
    if not stock_code or pd.isna(stock_code):
        return None
    
    # 文字列に変換して整形
    stock_code = str(stock_code).strip()
    
    # 4桁の数字でない場合はスキップ
    if not stock_code.isdigit() or len(stock_code) != 4:
        return None
    
    ticker = f"{stock_code}.T"
    
    for attempt in range(retry_count):
        try:
            stock = yf.Ticker(ticker)
            info = stock.info
            market_cap = info.get('marketCap')
            
            if market_cap:
                return market_cap
            
            # marketCapがない場合、株価×株式数で計算を試みる
            hist = stock.history(period="1d")
            if not hist.empty:
                current_price = hist['Close'].iloc[-1]
                shares_outstanding = info.get('sharesOutstanding')
                if current_price and shares_outstanding:
                    return current_price * shares_outstanding
            
        except Exception as e:
            if attempt < retry_count - 1:
                time.sleep(1)
                continue
    
    return None

# 提出会社のリストを作成
filing_companies = df[['filing_company_code', 'filing_company_name', 'filing_stock_code']].drop_duplicates()
print(f"時価総額を取得する提出会社数: {len(filing_companies)}")

# 時価総額を取得
market_caps = []
for idx, row in filing_companies.iterrows():
    if idx % 10 == 0:
        print(f"処理中... {idx}/{len(filing_companies)}")
    
    market_cap = get_market_cap(row['filing_stock_code'])
    market_caps.append({
        'filing_company_code': row['filing_company_code'],
        'filing_company_name': row['filing_company_name'],
        'filing_stock_code': row['filing_stock_code'],
        'filing_market_cap': market_cap
    })
    
    # API制限対策
    if idx > 0 and idx % 20 == 0:
        time.sleep(2)

market_cap_df = pd.DataFrame(market_caps)
print(f"\n時価総額取得完了: {market_cap_df['filing_market_cap'].notna().sum()}/{len(market_cap_df)} 社")

## 4. 保有株式の現在価値を計算

In [ ]:
def get_current_stock_price(stock_code, retry_count=3):
    """株式コードから現在の株価を取得"""
    if not stock_code or pd.isna(stock_code):
        return None
    
    stock_code = str(stock_code).strip()
    if not stock_code.isdigit() or len(stock_code) != 4:
        return None
    
    ticker = f"{stock_code}.T"
    
    for attempt in range(retry_count):
        try:
            stock = yf.Ticker(ticker)
            hist = stock.history(period="1d")
            if not hist.empty:
                return hist['Close'].iloc[-1]
        except Exception:
            if attempt < retry_count - 1:
                time.sleep(0.5)
                continue
    
    return None

# 保有株式の現在価値を計算
print("保有株式の現在価値を計算中...")

# 重複する銘柄の株価取得を効率化
unique_held_stocks = df[df['held_stock_code'].notna()]['held_stock_code'].unique()
print(f"ユニークな保有銘柄数: {len(unique_held_stocks)}")

# 株価辞書を作成
stock_prices = {}
for i, stock_code in enumerate(unique_held_stocks):
    if i % 20 == 0:
        print(f"株価取得中... {i}/{len(unique_held_stocks)}")
    
    price = get_current_stock_price(stock_code)
    if price:
        stock_prices[stock_code] = price
    
    # API制限対策
    if i > 0 and i % 30 == 0:
        time.sleep(2)

print(f"株価取得完了: {len(stock_prices)}/{len(unique_held_stocks)} 銘柄")

# 現在価値を計算
df['current_stock_price'] = df['held_stock_code'].map(stock_prices)
df['current_market_value'] = df['held_shares'] * df['current_stock_price']

# 現在価値を百万円単位に変換
df['current_market_value_million'] = df['current_market_value'] / 1_000_000

print(f"\n現在価値計算完了: {df['current_market_value'].notna().sum()}/{len(df)} レコード")

## 5. 時価総額比率の計算

In [ ]:
# 提出会社ごとの保有株式の合計時価総額を計算
holdings_summary = df.groupby(['filing_company_code', 'filing_company_name']).agg({
    'book_value_million_yen': 'sum',  # 貸借対照表計上額の合計
    'current_market_value': 'sum',     # 現在価値の合計
    'held_security_name': 'count'      # 保有銘柄数
}).reset_index()

holdings_summary.columns = ['filing_company_code', 'filing_company_name', 
                           'total_book_value_million', 'total_current_value', 'holdings_count']

# 時価総額データとマージ
result_df = pd.merge(holdings_summary, market_cap_df, on=['filing_company_code', 'filing_company_name'], how='left')

# 時価総額比率を計算
result_df['market_cap_ratio'] = result_df['total_current_value'] / result_df['filing_market_cap'] * 100

# 有効なデータのみフィルタリング
valid_result_df = result_df[result_df['market_cap_ratio'].notna() & (result_df['filing_market_cap'] > 0)]

# 時価総額比率で降順ソート
valid_result_df = valid_result_df.sort_values('market_cap_ratio', ascending=False)

print(f"時価総額比率計算完了: {len(valid_result_df)} 社")
print(f"\n=== 時価総額比率 上位20社 ===")

# 表示用に整形
display_df = valid_result_df.head(20).copy()
display_df['filing_market_cap_billion'] = display_df['filing_market_cap'] / 1e9
display_df['total_current_value_billion'] = display_df['total_current_value'] / 1e9

# 表示
display_columns = ['filing_company_name', 'filing_stock_code', 'market_cap_ratio', 
                  'filing_market_cap_billion', 'total_current_value_billion', 'holdings_count']
display_df_formatted = display_df[display_columns].copy()
display_df_formatted.columns = ['会社名', '証券コード', '時価総額比率(%)', 
                               '会社時価総額(十億円)', '保有株式時価総額(十億円)', '保有銘柄数']

# 数値を整形
display_df_formatted['時価総額比率(%)'] = display_df_formatted['時価総額比率(%)'].round(2)
display_df_formatted['会社時価総額(十億円)'] = display_df_formatted['会社時価総額(十億円)'].round(1)
display_df_formatted['保有株式時価総額(十億円)'] = display_df_formatted['保有株式時価総額(十億円)'].round(1)

display(display_df_formatted)

## 6. 分析結果のビジュアライゼーション

In [ ]:
# グラフ作成
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. 時価総額比率の分布
axes[0, 0].hist(valid_result_df['market_cap_ratio'], bins=30, alpha=0.7, color='skyblue', edgecolor='black')
axes[0, 0].set_title('Distribution of Market Cap Ratio', fontsize=14)
axes[0, 0].set_xlabel('Market Cap Ratio (%)', fontsize=12)
axes[0, 0].set_ylabel('Frequency', fontsize=12)
axes[0, 0].axvline(valid_result_df['market_cap_ratio'].median(), color='red', linestyle='--', 
                   label=f'Median: {valid_result_df["market_cap_ratio"].median():.1f}%')
axes[0, 0].legend()

# 2. 時価総額比率上位20社
top_20 = valid_result_df.head(20)
y_pos = np.arange(len(top_20))
axes[0, 1].barh(y_pos, top_20['market_cap_ratio'], color='lightgreen', edgecolor='darkgreen')
axes[0, 1].set_yticks(y_pos)
axes[0, 1].set_yticklabels([name[:20] + '...' if len(name) > 20 else name 
                           for name in top_20['filing_company_name']], fontsize=8)
axes[0, 1].set_title('Top 20 Companies by Market Cap Ratio', fontsize=14)
axes[0, 1].set_xlabel('Market Cap Ratio (%)', fontsize=12)

# 3. 会社時価総額 vs 保有株式時価総額（散布図）
axes[1, 0].scatter(valid_result_df['filing_market_cap'] / 1e9, 
                  valid_result_df['total_current_value'] / 1e9, 
                  alpha=0.6, color='coral', edgecolor='darkred')
axes[1, 0].set_title('Company Market Cap vs Holdings Value', fontsize=14)
axes[1, 0].set_xlabel('Company Market Cap (Billion Yen)', fontsize=12)
axes[1, 0].set_ylabel('Holdings Market Value (Billion Yen)', fontsize=12)
axes[1, 0].set_xscale('log')
axes[1, 0].set_yscale('log')

# 対角線を追加
max_val = max(axes[1, 0].get_xlim()[1], axes[1, 0].get_ylim()[1])
axes[1, 0].plot([0.1, max_val], [0.1, max_val], 'k--', alpha=0.3, label='1:1 line')
axes[1, 0].legend()

# 4. 保有銘柄数 vs 時価総額比率
axes[1, 1].scatter(valid_result_df['holdings_count'], 
                  valid_result_df['market_cap_ratio'], 
                  alpha=0.6, color='gold', edgecolor='darkorange')
axes[1, 1].set_title('Number of Holdings vs Market Cap Ratio', fontsize=14)
axes[1, 1].set_xlabel('Number of Holdings', fontsize=12)
axes[1, 1].set_ylabel('Market Cap Ratio (%)', fontsize=12)

plt.tight_layout()
plt.show()

# 統計情報
print("\n=== 統計情報 ===")
print(f"時価総額比率の平均: {valid_result_df['market_cap_ratio'].mean():.2f}%")
print(f"時価総額比率の中央値: {valid_result_df['market_cap_ratio'].median():.2f}%")
print(f"時価総額比率の標準偏差: {valid_result_df['market_cap_ratio'].std():.2f}%")
print(f"時価総額比率の最大値: {valid_result_df['market_cap_ratio'].max():.2f}%")
print(f"時価総額比率が10%を超える企業数: {(valid_result_df['market_cap_ratio'] > 10).sum()}社")
print(f"時価総額比率が20%を超える企業数: {(valid_result_df['market_cap_ratio'] > 20).sum()}社")

## 7. 詳細分析

In [ ]:
# 時価総額比率が高い企業の詳細分析
high_ratio_companies = valid_result_df[valid_result_df['market_cap_ratio'] > 20]

if len(high_ratio_companies) > 0:
    print(f"=== 時価総額比率が20%を超える企業の詳細 ===")
    print(f"該当企業数: {len(high_ratio_companies)}社\n")
    
    for idx, company in high_ratio_companies.iterrows():
        print(f"【{company['filing_company_name']}】")
        print(f"  証券コード: {company['filing_stock_code']}")
        print(f"  時価総額比率: {company['market_cap_ratio']:.1f}%")
        print(f"  会社時価総額: {company['filing_market_cap']/1e9:.1f} 十億円")
        print(f"  保有株式時価総額: {company['total_current_value']/1e9:.1f} 十億円")
        print(f"  保有銘柄数: {company['holdings_count']}")
        
        # この企業の保有銘柄トップ5を表示
        company_holdings = df[df['filing_company_code'] == company['filing_company_code']]
        top_holdings = company_holdings.nlargest(5, 'current_market_value_million')
        
        if len(top_holdings) > 0:
            print("  主要保有銘柄:")
            for _, holding in top_holdings.iterrows():
                print(f"    - {holding['held_security_name']}: {holding['current_market_value_million']:.1f} 百万円")
        print()

# 業種別分析（もし業種情報があれば）
if 'sector' in df.columns:
    sector_analysis = df.groupby('sector').agg({
        'filing_company_code': 'nunique',
        'current_market_value': 'sum'
    }).reset_index()
    sector_analysis.columns = ['sector', 'company_count', 'total_holdings_value']
    sector_analysis = sector_analysis.sort_values('total_holdings_value', ascending=False)
    
    print("\n=== 業種別保有株式時価総額 ===")
    display(sector_analysis.head(10))

## 8. 結果をエクスポート

In [ ]:
# 分析結果をCSVに保存
output_filename = f"market_cap_ratio_analysis_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"

# 保存用データフレームを作成
export_df = valid_result_df[[
    'filing_company_code', 'filing_company_name', 'filing_stock_code',
    'filing_market_cap', 'total_current_value', 'total_book_value_million',
    'market_cap_ratio', 'holdings_count'
]].copy()

# カラム名を日本語に変更
export_df.columns = [
    'EDINETコード', '会社名', '証券コード',
    '会社時価総額', '保有株式時価総額', '保有株式簿価総額(百万円)',
    '時価総額比率(%)', '保有銘柄数'
]

# CSVに保存
export_df.to_csv(output_filename, index=False, encoding='utf-8-sig')
print(f"分析結果を保存しました: {output_filename}")

# サマリーレポートも作成
summary_filename = f"market_cap_ratio_summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

with open(summary_filename, 'w', encoding='utf-8') as f:
    f.write("=== 特定投資株式の時価総額比率分析サマリー ===\n\n")
    f.write(f"分析日時: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"分析対象企業数: {len(valid_result_df)}社\n")
    f.write(f"\n【統計情報】\n")
    f.write(f"時価総額比率の平均: {valid_result_df['market_cap_ratio'].mean():.2f}%\n")
    f.write(f"時価総額比率の中央値: {valid_result_df['market_cap_ratio'].median():.2f}%\n")
    f.write(f"時価総額比率の標準偏差: {valid_result_df['market_cap_ratio'].std():.2f}%\n")
    f.write(f"時価総額比率の最大値: {valid_result_df['market_cap_ratio'].max():.2f}%\n")
    f.write(f"\n【閾値別企業数】\n")
    f.write(f"時価総額比率が10%を超える企業: {(valid_result_df['market_cap_ratio'] > 10).sum()}社\n")
    f.write(f"時価総額比率が20%を超える企業: {(valid_result_df['market_cap_ratio'] > 20).sum()}社\n")
    f.write(f"時価総額比率が30%を超える企業: {(valid_result_df['market_cap_ratio'] > 30).sum()}社\n")
    f.write(f"\n【時価総額比率 上位10社】\n")
    
    for i, (_, company) in enumerate(valid_result_df.head(10).iterrows(), 1):
        f.write(f"{i}. {company['filing_company_name']} ({company['filing_stock_code']}): {company['market_cap_ratio']:.1f}%\n")

print(f"サマリーレポートを保存しました: {summary_filename}")

print("\n=== 処理完了 ===")
print(f"時価総額比率の計算と分析が完了しました。")
print(f"最も高い時価総額比率: {valid_result_df.iloc[0]['filing_company_name']} - {valid_result_df.iloc[0]['market_cap_ratio']:.1f}%")

## 9. インタラクティブな追加分析（オプション）

In [ ]:
# 特定の企業の詳細を確認
def analyze_company(company_name_or_code):
    """特定企業の保有株式詳細を分析"""
    # 会社名または証券コードで検索
    company_data = valid_result_df[
        (valid_result_df['filing_company_name'].str.contains(company_name_or_code, na=False)) |
        (valid_result_df['filing_stock_code'].astype(str) == str(company_name_or_code))
    ]
    
    if company_data.empty:
        print(f"該当する企業が見つかりません: {company_name_or_code}")
        return
    
    company = company_data.iloc[0]
    print(f"\n=== {company['filing_company_name']} の分析 ===")
    print(f"証券コード: {company['filing_stock_code']}")
    print(f"時価総額: {company['filing_market_cap']/1e9:.1f} 十億円")
    print(f"保有株式時価総額: {company['total_current_value']/1e9:.1f} 十億円")
    print(f"時価総額比率: {company['market_cap_ratio']:.2f}%")
    print(f"保有銘柄数: {company['holdings_count']}")
    
    # 保有銘柄の詳細
    holdings = df[df['filing_company_code'] == company['filing_company_code']].copy()
    holdings = holdings.sort_values('current_market_value', ascending=False)
    
    print(f"\n【保有銘柄詳細（上位10銘柄）】")
    for i, (_, holding) in enumerate(holdings.head(10).iterrows(), 1):
        print(f"{i}. {holding['held_security_name']}")
        print(f"   株式数: {holding['held_shares']:,}")
        print(f"   簿価: {holding['book_value_million_yen']:.1f} 百万円")
        if pd.notna(holding['current_market_value_million']):
            print(f"   時価: {holding['current_market_value_million']:.1f} 百万円")
            print(f"   評価損益: {(holding['current_market_value_million'] - holding['book_value_million_yen']):.1f} 百万円")
        print()

# 使用例
# analyze_company("トヨタ")  # 会社名で検索
# analyze_company("7203")    # 証券コードで検索

print("特定企業の詳細分析を行うには、analyze_company('会社名または証券コード') を実行してください。")